# Heart Disease Prediction Using Machine Learning

**Dataset:** UCI Heart Disease (Cleveland subset)  
**Goal:** Binary classification — predict whether a patient has heart disease (1) or not (0)  
**Models:** Logistic Regression, K-Nearest Neighbours, Decision Tree, Random Forest  

---

## Section 1 — Imports & Setup

We import all the libraries we need at the top of the notebook so everything is available throughout.

- **pandas / numpy** — data manipulation and numerical operations
- **matplotlib / seaborn** — data visualisation
- **scikit-learn** — machine learning models, preprocessing, and evaluation

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')   # suppress minor warnings for cleaner output

# ── Data handling ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn: preprocessing ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Scikit-learn: models ──────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# ── Scikit-learn: evaluation ──────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── Global settings ───────────────────────────────────────────────────────────
RANDOM_STATE = 42           # ensures reproducible results every time we run
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid')   # clean plot background
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully!')

---
## Section 2 — Data Loading

We load the UCI Heart Disease (Cleveland) dataset from a CSV file.

**Feature descriptions:**

| Column | Description |
|--------|-------------|
| `age` | Age in years |
| `sex` | 1 = male, 0 = female |
| `cp` | Chest pain type (0 = typical angina … 3 = asymptomatic) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl (1 = true, 0 = false) |
| `restecg` | Resting ECG results (0–2) |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina (1 = yes, 0 = no) |
| `oldpeak` | ST depression induced by exercise relative to rest |
| `slope` | Slope of peak exercise ST segment (0–2) |
| `ca` | Number of major vessels coloured by fluoroscopy (0–3) |
| `thal` | Thalassemia (1 = normal, 2 = fixed defect, 3 = reversible defect) |
| `target` | **1 = heart disease present, 0 = no heart disease** |

In [ ]:
# Load the dataset into a pandas DataFrame
df = pd.read_csv('heart.csv')

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nColumn names: {list(df.columns)}')

In [ ]:
# Preview the first five rows
df.head()

---
## Section 3 — Data Exploration

Before building any model we need to understand what the data looks like:
- **Data types** — are the columns numeric or categorical?
- **Value ranges** — are there any suspicious min/max values?
- **Target balance** — is the dataset roughly balanced between the two classes?

In [ ]:
# Data types and non-null counts for each column
df.info()

In [ ]:
# Statistical summary: count, mean, std, min, quartiles, max
df.describe().round(2)

In [ ]:
# How many patients have/don't have heart disease?
print('Target value counts:')
print(df['target'].value_counts())
print(f'\nClass balance: {df["target"].value_counts(normalize=True).round(3).to_dict()}')

---
## Section 4 — Missing Value Analysis

Missing values can cause errors during model training. We check every column for nulls.

The UCI Cleveland dataset is already clean — no missing values. If there were missing values, common strategies are:
- **Numeric columns** → fill with the column mean or median
- **Categorical columns** → fill with the most frequent value (mode)
- **Drop rows** if very few rows are affected

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Visualise missing values as a heatmap (yellow = missing, blue = present)
plt.figure(figsize=(14, 3))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Value Heatmap (yellow = missing)', fontsize=13)
plt.tight_layout()
plt.show()

print('No missing values found — dataset is complete.')

---
## Section 5 — Exploratory Data Analysis (EDA)

EDA helps us understand the distribution of features, relationships between variables, and patterns that may help the model. Good EDA often reveals which features are most important.

In [ ]:
# ── 5.1  Target class distribution ───────────────────────────────────────────
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='target', data=df, palette=['#4C72B0', '#DD8452'])
ax.set_xticks([0, 1])
ax.set_xticklabels(['No Heart Disease (0)', 'Heart Disease (1)'])
plt.title('Target Class Distribution', fontsize=13)
plt.ylabel('Count')

# Add count labels on top of each bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2  Correlation heatmap ──────────────────────────────────────────────────
# Correlation tells us how strongly each feature is linearly related to
# every other feature. Values close to +1 or -1 indicate strong relationships.
plt.figure(figsize=(12, 9))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # hide upper triangle
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3  Age distribution split by target ────────────────────────────────────
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='age', hue='target', bins=20,
             palette=['#4C72B0', '#DD8452'], kde=True)
plt.title('Age Distribution by Heart Disease Status', fontsize=13)
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(title='Target', labels=['No Disease', 'Disease'])
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4  Histograms for all numeric features ──────────────────────────────────
# These help us see the shape of each feature's distribution.
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(18, 4))
for ax, col in zip(axes, numeric_cols):
    ax.hist(df[col], bins=20, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('Distribution of Continuous Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.5  Count plots for categorical features vs target ───────────────────────
# These show how each category relates to the presence/absence of heart disease.
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, categorical_cols):
    sns.countplot(x=col, hue='target', data=df,
                  palette=['#4C72B0', '#DD8452'], ax=ax)
    ax.set_title(f'{col} vs target')
    ax.legend(title='Target', labels=['No Disease', 'Disease'], fontsize=8)

plt.suptitle('Categorical Features vs Target', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.6  Box plots: continuous features by target ────────────────────────────
# Box plots show the median, spread, and outliers for each group.
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(18, 5))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x='target', y=col, data=df,
                palette=['#4C72B0', '#DD8452'], ax=ax)
    ax.set_xticklabels(['No Disease', 'Disease'])
    ax.set_title(col)
plt.suptitle('Continuous Features by Heart Disease Status', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## Section 6 — Data Preprocessing

Before feeding data to a machine learning model we need to:
1. **Separate features (X) from the target (y)**
2. **Scale continuous features** — algorithms like Logistic Regression and KNN are sensitive to the scale of features. We use `StandardScaler` which transforms each feature to have mean = 0 and standard deviation = 1.

> **Note:** Categorical columns in this dataset are already encoded as integers, so no additional encoding is needed.

In [ ]:
# ── 6.1  Separate features and target ────────────────────────────────────────
X = df.drop('target', axis=1)   # all columns except the target
y = df['target']                 # the column we want to predict

print(f'Features shape : {X.shape}')   # (303, 13)
print(f'Target shape   : {y.shape}')   # (303,)
print(f'\nFeature columns: {list(X.columns)}')

In [ ]:
# ── 6.2  Train / Test Split ───────────────────────────────────────────────────
# We split the data BEFORE scaling to prevent data leakage.
# Data leakage = letting test-set information influence the training process.
#
# stratify=y  →  ensures both splits have the same class ratio as the full dataset
# test_size=0.2 → 80% training, 20% testing

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set   : {X_train.shape[0]} samples')
print(f'Test set       : {X_test.shape[0]} samples')
print(f'\nTraining target distribution:\n{y_train.value_counts()}')
print(f'\nTest target distribution:\n{y_test.value_counts()}')

In [ ]:
# ── 6.3  Feature Scaling ──────────────────────────────────────────────────────
# StandardScaler: transforms features so that mean=0 and std=1.
# Rule: fit the scaler ONLY on training data, then use it to transform both
# training and test data. This prevents test data from influencing the scaling.

scaler = StandardScaler()

# fit_transform: learns the mean/std from training data AND transforms it
X_train_scaled = scaler.fit_transform(X_train)

# transform only: applies the SAME mean/std learned from training data
X_test_scaled  = scaler.transform(X_test)

print('Feature scaling complete.')
print(f'Scaled training set  — mean ≈ {X_train_scaled.mean():.4f}, std ≈ {X_train_scaled.std():.4f}')
print(f'Scaled test set      — mean ≈ {X_test_scaled.mean():.4f},  std ≈ {X_test_scaled.std():.4f}')

---
## Section 7 — Model Training

We train four different classification algorithms and store them in a dictionary for easy comparison.

| Model | Why we use it |
|-------|---------------|
| **Logistic Regression** | Simple, interpretable baseline; good for linearly separable data |
| **K-Nearest Neighbours** | Non-parametric; classifies based on the K closest training points |
| **Decision Tree** | Rule-based; highly interpretable; can overfit without depth limits |
| **Random Forest** | Ensemble of decision trees; generally robust and accurate |

In [ ]:
# ── Define the four models ────────────────────────────────────────────────────

# Logistic Regression
# max_iter=1000 ensures the optimiser has enough iterations to converge
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

# K-Nearest Neighbours
# n_neighbors=5 means: look at the 5 closest training points to make a decision
knn_model = KNeighborsClassifier(n_neighbors=5)

# Decision Tree
# max_depth=5 limits how deep the tree grows to reduce overfitting
dt_model = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)

# Random Forest
# n_estimators=100 means we build 100 decision trees and combine their votes
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# Store all models in a dictionary for easy looping later
models = {
    'Logistic Regression' : lr_model,
    'KNN'                 : knn_model,
    'Decision Tree'       : dt_model,
    'Random Forest'       : rf_model,
}

print('Models defined. Starting training...')

In [ ]:
# ── Train all models ──────────────────────────────────────────────────────────
# Logistic Regression and KNN are distance/gradient based → use SCALED data.
# Decision Tree and Random Forest are tree-based → scale doesn't matter,
# but we use scaled data for consistency.

for name, model in models.items():
    model.fit(X_train_scaled, y_train)   # .fit() = train the model
    print(f'  {name} — trained ✓')

---
## Section 8 — Model Evaluation

We evaluate each model on the **test set** (data the model has never seen) using:

- **Accuracy** = (correct predictions) / (total predictions)
- **Precision** = of all predicted positives, how many are actually positive
- **Recall** = of all actual positives, how many did we correctly find
- **F1-score** = harmonic mean of Precision and Recall (balances both)
- **Confusion Matrix** = table showing true positives, false positives, true negatives, false negatives

In [ ]:
# ── Helper function: evaluate one model and return its metrics ────────────────
def evaluate_model(name, model, X_test_data, y_test_data):
    """
    Prints a full evaluation report for the given model and
    returns a dictionary of the key metrics.
    """
    y_pred = model.predict(X_test_data)   # generate predictions on the test set

    acc  = accuracy_score(y_test_data, y_pred)
    prec = precision_score(y_test_data, y_pred)
    rec  = recall_score(y_test_data, y_pred)
    f1   = f1_score(y_test_data, y_pred)

    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(y_test_data, y_pred,
                                target_names=['No Disease', 'Disease']))

    # ── Confusion matrix heatmap ──────────────────────────────────────────────
    cm = confusion_matrix(y_test_data, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted: No Disease', 'Predicted: Disease'],
                yticklabels=['Actual: No Disease', 'Actual: Disease'])
    plt.title(f'Confusion Matrix — {name}', fontsize=12)
    plt.tight_layout()
    plt.show()

    return {'Model': name, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1-Score': f1}

print('Evaluation function defined.')

In [ ]:
# ── Evaluate all four models and collect results ──────────────────────────────
results = []

for name, model in models.items():
    metrics = evaluate_model(name, model, X_test_scaled, y_test)
    results.append(metrics)

In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('Model')
results_df = results_df.sort_values('F1-Score', ascending=False)
results_df = results_df.round(4)

print('\nModel Comparison (sorted by F1-Score):')
results_df

In [ ]:
# ── Bar chart: metric comparison across all models ────────────────────────────
results_plot = results_df.reset_index().melt(
    id_vars='Model', var_name='Metric', value_name='Score'
)

plt.figure(figsize=(12, 5))
sns.barplot(x='Model', y='Score', hue='Metric', data=results_plot)
plt.title('Model Performance Comparison', fontsize=13)
plt.ylim(0.5, 1.05)
plt.ylabel('Score')
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## Section 9 — Best Model Selection

We select the best model based on **F1-Score** because:
- In medical diagnosis, both **false positives** (wrongly diagnosing a healthy person) and **false negatives** (missing a sick person) are costly.
- F1-score balances Precision and Recall, making it more meaningful than accuracy alone for this problem.

In [ ]:
# Find the model with the highest F1-Score
best_model_name = results_df['F1-Score'].idxmax()
best_f1         = results_df.loc[best_model_name, 'F1-Score']
best_accuracy   = results_df.loc[best_model_name, 'Accuracy']

print(f'Best Model   : {best_model_name}')
print(f'F1-Score     : {best_f1:.4f}')
print(f'Accuracy     : {best_accuracy:.4f}')
print(f'\nReasoning: "{best_model_name}" achieved the highest F1-Score ({best_f1:.4f}),'
      f' meaning it has the best balance between correctly identifying'
      f' heart disease cases and avoiding false alarms.')

# Retrieve the trained model object for the best model
best_model = models[best_model_name]

---
## Section 10 — Prediction Function

We create a simple, reusable function that:
1. Accepts patient data as a Python dictionary
2. Applies the same scaling used during training
3. Uses the best model to make a prediction
4. Returns a human-readable result with probability

In [ ]:
# ── Feature column order (must match training data exactly) ───────────────────
FEATURE_COLUMNS = list(X.columns)

def predict_heart_disease(patient_data: dict) -> str:
    """
    Predicts whether a patient has heart disease.

    Parameters
    ----------
    patient_data : dict
        A dictionary with the 13 feature keys:
        age, sex, cp, trestbps, chol, fbs, restecg,
        thalach, exang, oldpeak, slope, ca, thal

    Returns
    -------
    str
        Prediction result with probability.

    Example
    -------
    >>> predict_heart_disease({'age':52,'sex':1,'cp':0,'trestbps':125,
    ...     'chol':212,'fbs':0,'restecg':1,'thalach':168,'exang':0,
    ...     'oldpeak':1.0,'slope':2,'ca':2,'thal':3})
    """
    # Step 1: Convert the dictionary to a DataFrame with one row
    #         The column order must match the training data exactly.
    input_df = pd.DataFrame([patient_data])[FEATURE_COLUMNS]

    # Step 2: Apply the SAME scaler fitted on the training data
    input_scaled = scaler.transform(input_df)

    # Step 3: Make the prediction (0 or 1)
    prediction = best_model.predict(input_scaled)[0]

    # Step 4: Get the probability of the positive class (heart disease)
    probability = best_model.predict_proba(input_scaled)[0][1]

    # Step 5: Return a human-readable result
    if prediction == 1:
        return (f'[!] HEART DISEASE DETECTED\n'
                f'    Model: {best_model_name}\n'
                f'    Probability of heart disease: {probability:.2%}\n'
                f'    Recommendation: Please consult a cardiologist.')
    else:
        return (f'[OK] NO HEART DISEASE DETECTED\n'
                f'     Model: {best_model_name}\n'
                f'     Probability of heart disease: {probability:.2%}\n'
                f'     Recommendation: Maintain a healthy lifestyle.')


print('predict_heart_disease() function is ready.')

---
## Section 11 — Sample Predictions

We test the prediction function with two real examples from the test set so you can verify it works correctly.

In [ ]:
# ── Example 1: take the first patient from the test set ───────────────────────
sample_1 = X_test.iloc[0].to_dict()   # real patient data
actual_1  = y_test.iloc[0]            # what the label actually says

print('Patient 1 — Input Features:')
for k, v in sample_1.items():
    print(f'  {k:10s}: {v}')
print(f'\nActual label  : {actual_1} ({"Heart Disease" if actual_1 == 1 else "No Heart Disease"})')
print(f'\nModel Result  :')
print(predict_heart_disease(sample_1))

In [ ]:
# ── Example 2: take another patient from the test set ─────────────────────────
sample_2 = X_test.iloc[5].to_dict()
actual_2  = y_test.iloc[5]

print('Patient 2 — Input Features:')
for k, v in sample_2.items():
    print(f'  {k:10s}: {v}')
print(f'\nActual label  : {actual_2} ({"Heart Disease" if actual_2 == 1 else "No Heart Disease"})')
print(f'\nModel Result  :')
print(predict_heart_disease(sample_2))

In [ ]:
# ── Example 3: manually constructed patient ───────────────────────────────────
# You can change these values and re-run the cell to test different patients.
custom_patient = {
    'age'      : 55,   # age in years
    'sex'      : 1,    # 1 = male
    'cp'       : 0,    # 0 = typical angina (most concerning chest pain type)
    'trestbps' : 140,  # resting blood pressure (mm Hg)
    'chol'     : 250,  # cholesterol (mg/dl)
    'fbs'      : 1,    # fasting blood sugar > 120 mg/dl
    'restecg'  : 1,    # resting ECG: normal
    'thalach'  : 150,  # max heart rate
    'exang'    : 1,    # exercise-induced angina: yes
    'oldpeak'  : 2.0,  # ST depression
    'slope'    : 1,    # slope of peak ST segment
    'ca'       : 1,    # number of major vessels
    'thal'     : 2,    # thalassemia: fixed defect
}

print('Custom Patient — Input Features:')
for k, v in custom_patient.items():
    print(f'  {k:10s}: {v}')
print(f'\nModel Result  :')
print(predict_heart_disease(custom_patient))

---
## Summary

| Step | What we did |
|------|-------------|
| Data Loading | Loaded 303 patient records with 13 features from the UCI Heart Disease dataset |
| Exploration | Checked shapes, types, value ranges, and class balance |
| Missing Values | Confirmed no missing values; explained strategies for handling them |
| EDA | Visualised distributions, correlations, and feature vs. target relationships |
| Preprocessing | Separated X/y, performed 80/20 stratified split, applied StandardScaler |
| Model Training | Trained Logistic Regression, KNN, Decision Tree, Random Forest |
| Evaluation | Computed Accuracy, Precision, Recall, F1-Score, and Confusion Matrix for each model |
| Best Model | Selected the model with the highest F1-Score |
| Prediction | Built `predict_heart_disease()` function for real-world usage |

---
*Heart Disease Prediction — Academic Internship Project*